# Healthcare admission

## Dataset

We use the ACSPublicCoverage dataset from the Folktables benchmark, which is derived from the U.S. Census American Community Survey (ACS).

The task is a binary classification problem where the goal is to predict whether a person has public health insurance coverage (1) or not (0). Public coverage includes programs such as Medicaid and Medicare, and it serves as a proxy for access to healthcare services.

The input features include demographic and socioeconomic variables such as age, education, marital status, employment characteristics, and income-related attributes.

The dataset also contains sensitive attributes — race and sex — which are used to evaluate fairness in healthcare access.

### Race (RAC1P)

The variable RAC1P encodes a person’s self-identified race in the U.S. Census ACS data.

- 1: White alone
- 2: Black or African American alone
- 3: American Indian or Alaska Native alone
- 4: Alaska Native alone
- 5: American Indian alone
- 6: Asian alone
- 7: Native Hawaiian or Other Pacific Islander alone
- 8: Some other race alone
- 9: Two or more races

### Sex (SEX)

The variable SEX is binary in the ACS:

- 1: Male
- 2: Female

## Models

We train two logistic regression models on the same ACSPublicCoverage dataset.

### Accuracy-first model

This model uses all available features, including sensitive attributes such as race and sex, and is optimized purely for predictive performance (accuracy and ROC AUC). It represents a standard automated ML pipeline that prioritizes performance without fairness constraints.

### Fairness-aware model
This model excludes race and sex from the input features in order to reduce direct discrimination. Although this may slightly reduce predictive accuracy, it aims to decrease disparities in healthcare access predictions between protected and non-protected groups.

Both models are evaluated not only by predictive performance, but also by fairness metrics, including:
Demographic Parity (differences in predicted probability of coverage across groups)
Equal Opportunity (differences in true positive rates across race and sex)
By comparing these two models, we analyze the trade-off between accuracy and fairness in automated healthcare decision systems.

## Data download

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
from folktables import ACSDataSource, ACSPublicCoverage

# Load ACS data
data_source = ACSDataSource(survey_year="2018", horizon="1-Year", survey="person")
acs_data = data_source.get_data(states=["CA"], download=True)

# Define task
task = ACSPublicCoverage

# Get filtered dataframe used by the task
acs_df, y, _ = task.df_to_pandas(acs_data)

# Build modeling dataframe
df = acs_df[task.features].copy()
df["target"] = y

# Add sensitive attributes
df["race"] = acs_df["RAC1P"].to_numpy()
df["sex"]  = acs_df["SEX"].to_numpy()

# Define feature columns
feature_names = [c for c in df.columns if c not in ["target", "race", "sex"]]

X = df[feature_names]
y = df["target"].astype(int)

## Train/Test split

In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Scale
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# Model 1: Accuracy-first

In [3]:
model_acc = LogisticRegression(max_iter=2000)
model_acc.fit(X_train_s, y_train)

y_pred = model_acc.predict(X_test_s)
y_prob = model_acc.predict_proba(X_test_s)[:, 1]

print("Accuracy (Healthcare baseline):", accuracy_score(y_test, y_pred))
print("ROC AUC (Healthcare baseline):", roc_auc_score(y_test, y_prob))

Accuracy (Healthcare baseline): 0.6854957057281016
ROC AUC (Healthcare baseline): 0.6794979303597883


## Fairness Metrics 

In [4]:
test_df = df.loc[X_test.index, ["race", "sex"]].copy()
test_df["prob"] = y_prob

print("\nAvg predicted probability by race:")
print(test_df.groupby("race")["prob"].mean())

print("\nAvg predicted probability by sex:")
print(test_df.groupby("sex")["prob"].mean())

race_dp_gap = test_df.groupby("race")["prob"].mean().max() - test_df.groupby("race")["prob"].mean().min()
sex_dp_gap  = test_df.groupby("sex")["prob"].mean().max()  - test_df.groupby("sex")["prob"].mean().min()

print("\nDemographic Parity gap (race):", race_dp_gap)
print("Demographic Parity gap (sex):", sex_dp_gap)


Avg predicted probability by race:
race
1.0    0.361764
2.0    0.426367
3.0    0.410209
4.0    0.532604
5.0    0.388405
6.0    0.349250
7.0    0.368408
8.0    0.386014
9.0    0.380657
Name: prob, dtype: float64

Avg predicted probability by sex:
sex
1.0    0.386751
2.0    0.354743
Name: prob, dtype: float64

Demographic Parity gap (race): 0.18335414120492077
Demographic Parity gap (sex): 0.0320080905587104


# Model 2: Fairness-awareness

In [5]:
fair_features = [f for f in feature_names if f not in ["RAC1P", "SEX"]]

X_fair = df[fair_features]
y = df["target"].astype(int)

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fair, y, test_size=0.3, random_state=42, stratify=y
)

scaler_f = StandardScaler()
X_train_f_s = scaler_f.fit_transform(X_train_f)
X_test_f_s = scaler_f.transform(X_test_f)

model_fair = LogisticRegression(max_iter=2000)
model_fair.fit(X_train_f_s, y_train_f)

y_pred_f = model_fair.predict(X_test_f_s)
y_prob_f = model_fair.predict_proba(X_test_f_s)[:, 1]

print("Accuracy (Healthcare fairness-aware):", accuracy_score(y_test_f, y_pred_f))
print("ROC AUC (Healthcare fairness-aware):", roc_auc_score(y_test_f, y_prob_f))

Accuracy (Healthcare fairness-aware): 0.6846777491760291
ROC AUC (Healthcare fairness-aware): 0.6765669095371454


## Fairness Evaluation 

In [6]:
test_df_f = df.loc[X_test_f.index, ["race", "sex"]].copy()
test_df_f["prob"] = y_prob_f

print("\nAvg predicted probability by race (fairness-aware):")
print(test_df_f.groupby("race")["prob"].mean())

print("\nAvg predicted probability by sex (fairness-aware):")
print(test_df_f.groupby("sex")["prob"].mean())

race_dp_gap_f = (
    test_df_f.groupby("race")["prob"].mean().max()
    - test_df_f.groupby("race")["prob"].mean().min()
)

sex_dp_gap_f = (
    test_df_f.groupby("sex")["prob"].mean().max()
    - test_df_f.groupby("sex")["prob"].mean().min()
)

print("\nDemographic Parity gap (race, fairness-aware):", race_dp_gap_f)
print("Demographic Parity gap (sex, fairness-aware):", sex_dp_gap_f)


Avg predicted probability by race (fairness-aware):
race
1.0    0.366915
2.0    0.425374
3.0    0.409402
4.0    0.552299
5.0    0.381099
6.0    0.345564
7.0    0.359685
8.0    0.376077
9.0    0.367863
Name: prob, dtype: float64

Avg predicted probability by sex (fairness-aware):
sex
1.0    0.374134
2.0    0.364867
Name: prob, dtype: float64

Demographic Parity gap (race, fairness-aware): 0.20673458166420372
Demographic Parity gap (sex, fairness-aware): 0.009266942813886125
